In [1]:
import nest_asyncio 
nest_asyncio.apply() # 중첩을 허용: 이미 돌고 있는 루프 안에서 또 루프를 돌려도 괜찮게 규칙을 완화

# 데이터 로드

## 1. SimpleWebPageReader
- 웹페이지를 쉽게 가져오도록 도와주는 클래스

In [1]:
from llama_index.readers.web import SimpleWebPageReader

# 웹페이지 로드 (불러오기)
documents = SimpleWebPageReader().load_data(urls=['https://www.naver.com']) # urls가 복수인 걸 보면 한번에 여러 웹사이트를 열 수 있다는 걸 알 수 있다.

In [4]:
documents

[Document(id_='41003924-6084-450b-87a9-5832a2e1b56f', embedding=None, metadata={'url': 'https://www.naver.com'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='   <!doctype html> <html lang="ko" class="fzoom"> <head> <meta charset="utf-8"> <meta name="Referrer" content="origin"> <meta http-equiv="X-UA-Compatible" content="IE=edge"> <meta name="viewport" content="width=1190"> <title>NAVER</title> <meta name="apple-mobile-web-app-title" content="NAVER"/> <meta name="robots" content="index,nofollow"/> <meta name="description" content="네이버 메인에서 다양한 정보와 유용한 컨텐츠를 만나 보세요"/> <meta property="og:title" content="네이버"> <meta property="og:url" content="https://www.naver.com/"> <meta property="og:image" content="https://s.pstatic.net/static/www/mobile/edit/2016/0705/mobile_212852414260.png"> <meta property="og:description" content="네이버 메인에서 다양한

In [5]:
print(documents[0])

Doc ID: 41003924-6084-450b-87a9-5832a2e1b56f
Text: <!doctype html> <html lang="ko" class="fzoom"> <head> <meta
charset="utf-8"> <meta name="Referrer" content="origin"> <meta http-
equiv="X-UA-Compatible" content="IE=edge"> <meta name="viewport"
content="width=1190"> <title>NAVER</title> <meta name="apple-mobile-
web-app-title" content="NAVER"/> <meta name="robots"
content="index,nofollow"/> <meta ...


## 2. 유튜브 자막 가져오기

In [9]:
from youtube_transcript_api import YouTubeTranscriptApi
from llama_index.core import Document

def get_youtube_document(video_url):
    """유튜브 url을 받아 자막을 가져온 뒤 라마인덱스 document 객체로 반환"""
    try:
        video_id = '' #1. url에서 video id를 추출(?다음부터)
        if 'v=' in video_url:
            video_id = video_url.split('v=')[1].split('&')[0] #'v='기준으로 리스트 2개로 나눠짐; v=뒤에 parameter는 [1]으로 지정
    # 공유를 눌렀을 땐 다른 방식 -> https://youtu.be/JvDHxjIZGTA?si=pTBoDH9SGlVDTQEg
        elif 'youtu.be/' in video_url:
            video_id = video_url.split('youtu.be/')[1].split('?')[0]
        
        if not video_id: 
            print('비디오 id를 찾을 수 없습니다!')
            return[]
        #2. 자막 가져오기
        yt = YouTubeTranscriptApi()
        transcript_list = yt.list(video_id)

        #3. 언어 선택 (한국어, 영어 순; 없으면 자동생성 가능한 것)
        try:
            transcript = transcript_list.find_transcript(['ko','en'])
        except:
            # 첫번째로 잡히는 자막 사용
            transcript = next(iter(transcript_list))

        transcript_data = transcript.fetch()
        
        #4. 텍스트 추출 및 결합
        full_text = ' '.join([entry.text for entry in transcript_data])

        #5. 라마인덱스의 도큐먼트 객체 생성
        return [Document(text=full_text, metadata={'video_url':video_url, 'video_id':video_id})]
    
    except Exception as e:
        print(f'에러 발생 : {e}') # 에러가 나면 에러 메시지를 보여주고 정상 종료
        return []


documents = get_youtube_document('https://youtu.be/hAEfi_SKTEU?si=S7ROovUwriO4ZIU7')

print(documents)

[Document(id_='2a3e5954-ded8-4abb-931e-66b8c4f868ab', embedding=None, metadata={'video_url': 'https://youtu.be/hAEfi_SKTEU?si=S7ROovUwriO4ZIU7', 'video_id': 'hAEfi_SKTEU'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='I could go somewhere Maybe anywhere 내 직감은 늘 맞으니깐 Just feel it 손끝엔 나비 That\'s my living ring 내 걱정 NO! 난 나답게 더 빛나져 Feel the breeze 나만 들리는 소리 I can tell myself that I DO ME, I DO me right 그냥 I DO ME, I DO ma way 다른 그림들은 upside-down 결국 내가 가는 길이야 남 눈치s, Why do you care? I\'m just 달릴 거야 멋대로 Still, pinky 노을빛 날개를 달고 ‘이걸 해야 돼.’ ‘거길 가야 돼.’ 상관없죠, 난 \'내\'가 될 거에여 콩 무당벌레, That\'s my piercing I\'m screaming Yeah! 난, 나답게 더 잘해여 Feel the breeze 나만 들리는 소리 I can tell myself that I DO ME, I DO me right 그냥 I DO ME, I DO ma way 다른 그림들은 upside-down 결국 내가 가는 길이야 남 눈치s, Why do you care? I\'m just 달릴 거야 멋대로 Still, pinky 노을빛 날개를 달고 I run in t

In [10]:
print(documents[0].text)

I could go somewhere Maybe anywhere 내 직감은 늘 맞으니깐 Just feel it 손끝엔 나비 That's my living ring 내 걱정 NO! 난 나답게 더 빛나져 Feel the breeze 나만 들리는 소리 I can tell myself that I DO ME, I DO me right 그냥 I DO ME, I DO ma way 다른 그림들은 upside-down 결국 내가 가는 길이야 남 눈치s, Why do you care? I'm just 달릴 거야 멋대로 Still, pinky 노을빛 날개를 달고 ‘이걸 해야 돼.’ ‘거길 가야 돼.’ 상관없죠, 난 '내'가 될 거에여 콩 무당벌레, That's my piercing I'm screaming Yeah! 난, 나답게 더 잘해여 Feel the breeze 나만 들리는 소리 I can tell myself that I DO ME, I DO me right 그냥 I DO ME, I DO ma way 다른 그림들은 upside-down 결국 내가 가는 길이야 남 눈치s, Why do you care? I'm just 달릴 거야 멋대로 Still, pinky 노을빛 날개를 달고 I run in the air! 잠 못 드는 밤, 나 달빛 보며 Maybe, I'm 별난 아이 Maybe, 내 꿈 의심 들 때도 Keep it in your soul 내 상상보다 더 큰 나였어 겁 없는 my face, no need to chase Eyes closed and 들려 whispering "I do me, I do it my way!" I DO ME, I DO me right 그냥 I DO ME, I DO ma way 다른 그림들은 upside-down 결국 내가 가는 길이야 남 눈치s, Why do you care? I'm just 달릴 거야 멋대로 Still, pinky 노을빛 날개를 달고 I run in the air!


## 3. 위키피디아 읽어오기

In [12]:
from llama_index.readers.wikipedia import WikipediaReader

reader = WikipediaReader()

documents = reader.load_data(pages=['South Korea'])

ModuleNotFoundError: No module named 'llama_index.readers.wikipedia'

# 여러가지 종류의 노드파서 알아보기

- 노드파서: 라마인덱스에서 문서를 작은 단위의 노드로 변환하는 역할을 한다.

## 1. Simple Node Parser (Text Splitter)

In [3]:
from llama_index.core import Document
from llama_index.core.node_parser import TokenTextSplitter, SentenceSplitter

In [4]:
text = """
LlamaIndex는 대규모 언어 모델(LLM)을 사용하여 개인 데이터를 처리하는 데이터 프레임워크입니다. 이 프레임워크는 데이터 수집부터 처리, 검색까지 전체 과정을 지원합니다.

주요 구성 요소는 다음과 같습니다:
Documents는 원시 데이터를 표현하는 기본 단위입니다. 텍스트 파일, PDF, 웹페이지 등 다양한 소스의 데이터를 포함할 수 있습니다.
Nodes는 Documents를 더 작은 단위로 분할한 것으로, LLM이 효과적으로 처리할 수 있는 크기입니다.

데이터 처리 과정은 다음과 같습니다:
먼저 데이터 로더를 사용하여 Documents를 생성합니다. 그 다음 Node Parser를 통해 Documents를 Nodes로 분할합니다.
마지막으로 인덱스를 구축하여 효율적인 검색이 가능하도록 합니다.

LlamaIndex는 다양한 인덱스 유형을 제공합니다. VectorStoreIndex는 임베딩 기반 검색을, SummaryIndex는 요약 기반 검색을 지원합니다.
또한 하이브리드 검색, 재순위화 등 고급 검색 기능도 제공하여 검색 품질을 향상시킬 수 있습니다.
"""

- TokenTextSplitter: 토큰 단위로 텍스트를 분할하는 클래스
    - LLM의 토큰 제한을 고려할 때 유용하다.

In [5]:
token_splitter = TokenTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

token_nodes = token_splitter.split_text(text)

In [6]:
token_nodes

['LlamaIndex는 대규모 언어 모델(LLM)을 사용하여 개인 데이터를 처리하는 데이터 프레임워크입니다. 이 프레임워크는 데이터 수집부터 처리, 검색까지 전체 과정을 지원합니다.\n\n주요 구성 요소는 다음과 같습니다:\nDocuments는 원시 데이터를 표현하는 기본',
 '같습니다:\nDocuments는 원시 데이터를 표현하는 기본 단위입니다. 텍스트 파일, PDF, 웹페이지 등 다양한 소스의 데이터를 포함할 수 있습니다.\nNodes는 Documents를 더 작은 단위로 분할한 것으로, LLM이 효과적으로 처리할 수 있는 크기입니다.\n\n데이터 처리 과정은 다음과',
 '처리할 수 있는 크기입니다.\n\n데이터 처리 과정은 다음과 같습니다:\n먼저 데이터 로더를 사용하여 Documents를 생성합니다. 그 다음 Node Parser를 통해 Documents를 Nodes로 분할합니다.\n마지막으로 인덱스를 구축하여 효율적인 검색이 가능하도록 합니다.\n\nLlamaIndex는 다양한',
 '검색이 가능하도록 합니다.\n\nLlamaIndex는 다양한 인덱스 유형을 제공합니다. VectorStoreIndex는 임베딩 기반 검색을, SummaryIndex는 요약 기반 검색을 지원합니다.\n또한 하이브리드 검색, 재순위화 등 고급 검색 기능도 제공하여 검색',
 '등 고급 검색 기능도 제공하여 검색 품질을 향상시킬 수 있습니다.']

In [7]:
for i, node in enumerate(token_nodes, start=1):
    print()
    print(f'Node {i}: ')
    print(node)


Node 1: 
LlamaIndex는 대규모 언어 모델(LLM)을 사용하여 개인 데이터를 처리하는 데이터 프레임워크입니다. 이 프레임워크는 데이터 수집부터 처리, 검색까지 전체 과정을 지원합니다.

주요 구성 요소는 다음과 같습니다:
Documents는 원시 데이터를 표현하는 기본

Node 2: 
같습니다:
Documents는 원시 데이터를 표현하는 기본 단위입니다. 텍스트 파일, PDF, 웹페이지 등 다양한 소스의 데이터를 포함할 수 있습니다.
Nodes는 Documents를 더 작은 단위로 분할한 것으로, LLM이 효과적으로 처리할 수 있는 크기입니다.

데이터 처리 과정은 다음과

Node 3: 
처리할 수 있는 크기입니다.

데이터 처리 과정은 다음과 같습니다:
먼저 데이터 로더를 사용하여 Documents를 생성합니다. 그 다음 Node Parser를 통해 Documents를 Nodes로 분할합니다.
마지막으로 인덱스를 구축하여 효율적인 검색이 가능하도록 합니다.

LlamaIndex는 다양한

Node 4: 
검색이 가능하도록 합니다.

LlamaIndex는 다양한 인덱스 유형을 제공합니다. VectorStoreIndex는 임베딩 기반 검색을, SummaryIndex는 요약 기반 검색을 지원합니다.
또한 하이브리드 검색, 재순위화 등 고급 검색 기능도 제공하여 검색

Node 5: 
등 고급 검색 기능도 제공하여 검색 품질을 향상시킬 수 있습니다.


- SentenceSplitter: 문장 단위로 텍스트를 분할하여 자연스러운 문장 경계를 유지하면서 노드를 나눌 수 있다.

In [8]:
sentence_splitter = SentenceSplitter(
    chunk_size=100,
    chunk_overlap=20
)

sentence_nodes = sentence_splitter.split_text(text)

In [9]:
for i, node in enumerate(sentence_nodes, start=1):
    print()
    print(f'Node {i}: ')
    print(node)


Node 1: 
LlamaIndex는 대규모 언어 모델(LLM)을 사용하여 개인 데이터를 처리하는 데이터 프레임워크입니다. 이 프레임워크는 데이터 수집부터 처리, 검색까지 전체 과정을 지원합니다.

Node 2: 
주요 구성 요소는 다음과 같습니다:
Documents는 원시 데이터를 표현하는 기본 단위입니다. 텍스트 파일, PDF, 웹페이지 등 다양한 소스의 데이터를 포함할 수 있습니다.

Node 3: 
Nodes는 Documents를 더 작은 단위로 분할한 것으로, LLM이 효과적으로 처리할 수 있는 크기입니다.

데이터 처리 과정은 다음과 같습니다:
먼저 데이터 로더를 사용하여 Documents를 생성합니다. 그 다음 Node Parser를 통해 Documents를 Nodes로 분할합니다.

Node 4: 
그 다음 Node Parser를 통해 Documents를 Nodes로 분할합니다.
마지막으로 인덱스를 구축하여 효율적인 검색이 가능하도록 합니다.

LlamaIndex는 다양한 인덱스 유형을 제공합니다.

Node 5: 
VectorStoreIndex는 임베딩 기반 검색을, SummaryIndex는 요약 기반 검색을 지원합니다.
또한 하이브리드 검색, 재순위화 등 고급 검색 기능도 제공하여 검색 품질을 향상시킬 수 있습니다.


## 2. JSONNodeParser

In [10]:
from llama_index.core.node_parser import JSONNodeParser
from llama_index.core import SimpleDirectoryReader

json_docs = SimpleDirectoryReader(input_dir='./data', required_exts=['.json']).load_data()

parser = JSONNodeParser()
json_nodes = parser.get_nodes_from_documents(json_docs)

In [11]:
# JSONNode 개수
len(json_nodes)

2

In [12]:
# 첫 번째 노드의 텍스트 출력
json_nodes[0].text

'질문 RAG가 무엇인가요?\n답변 RAG는 Retrieval Augmented Generation의 약자로, 대규모 언어 모델의 환각 현상을 줄이고 최신 정보를 반영할 수 있는 기술입니다.\n참고자료 제목 RAG 논문\n참고자료 저자 홍길동\n참고자료 제목 RAG 구현 가이드\n참고자료 링크 https://example.com/rag'

In [13]:
# 두 번째 노드의 텍스트 출력
json_nodes[1].text

'질문 RAG의 장점은 무엇인가요?\n답변 RAG는 외부 지식을 활용하여 답변의 정확도를 높이고, 실시간으로 업데이트되는 정보를 반영할 수 있습니다.\n키워드 검색 기반 생성\n키워드 지식 증강\n키워드 최신 정보 반영\n사용사례 챗봇\n사용사례 질의응답 시스템\n사용사례 문서 요약'

In [14]:
# 첫 번째 노드의 메타데이터
json_nodes[0].metadata

{'file_path': '/Users/tommynoh/ai_class/llm_labs/data/json_example.json',
 'file_name': 'json_example.json',
 'file_type': 'application/json',
 'file_size': 815,
 'creation_date': '2026-05-28',
 'last_modified_date': '2026-05-28'}

# 임베딩과 인덱스

In [15]:
# api 키 가져오기
from dotenv import load_dotenv

load_dotenv()

True

In [16]:
from llama_index.core.evaluation import SemanticSimilarityEvaluator

evaluator = SemanticSimilarityEvaluator()

sentence1 = """요가는 신체와 마음 모두에게 다양한 이점을 제공합니다. 
유연성, 근력, 균형감을 향상시키는 동시에 스트레스, 불안, 통증을 줄여줍니다. 
요가는 또한 숙면, 심장 건강, 전반적인 웰빙을 증진시킵니다.
운동이나 스트레스 해소 방법을 찾고 있다면, 요가가 좋은 선택이 될 수 있습니다."""

sentence2 = """Yoga provides various benefits for both body and mind.
It improves flexibility, strength, and balance while reducing stress, anxiety, and pain.
Yoga also enhances sleep quality, heart health, and overall well-being.
If you're looking for exercise or a way to relieve stress, yoga can be a great choice."""

result = evaluator.evaluate(
    response=sentence1,
    reference=sentence2
)

In [18]:
# 유사도 점수
result.score

0.8384070942299976

In [20]:
# 통과 여부 --> 기본 유사도 임계값은 0.8로 하는 편이다.
result.passing # 83프로니까 True가 나옴

True

## 1. VectorStoreIndex

- 라마인덱스에서 가장 많이 사용하는 인덱스 유형
- 문서를 벡터(임베딩)형태로 저장하과, 검색하는데 특화되어 있다.
- 저장된 벡터들 사이의 유사도를 코사인 유사도로 측정하여 효율적인 검색을 지원한다.
- RAG 시스템 구축에 아주 적합하다.

In [22]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings
import os 
#from dotenv import load_dotenv. (위에 이미 있음)


In [23]:
load_dotenv() # .env파일에서 환경변수 불러온다.

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY') # OpenAI API키 설정

- Settings 클래스: 라마인덱스에서 프로그램 전체에서 사용하는 중요한 설정을 한 곳에서 관리하는 도구
    - 언어모델(LLM), 임베딩모델, 모델 처리 방식 등 쉽게 설정하고 변경할 수 있다.

In [24]:
# LLM 설정
Settings.llm = OpenAI(model='gpt-4o', temperature=0.5)

In [25]:
# 데이터 로드
documents = SimpleDirectoryReader('./data/pdf_sample2').load_data()

In [34]:
# 파일 이름 확인
for doc in documents:
    print(doc.metadata['file_name'])

NIPS-2017-attention-is-all-you-need-Paper.pdf


- VectorStoreIndex 클래스: 라마인덱스에서 문서의 벡터 표현을 저장하고 관리하는 핵심 구조
    - 이 인덱스는 문서를 임베딩 형태로 변환하여 저장, 유사도 기반의 빠른 검색이 가능해진다.

In [36]:
# 인덱스 생성 및 데이터 임베딩
index = VectorStoreIndex.from_documents(documents, show_progress=True)

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/373 [00:00<?, ?it/s]

- 쿼리(질의) 엔진: 사용자의 질문을 받아 인덱스에서 관련 정보를 검색, 그 정보를 바탕으로 정확하고 관련성 높은 답변을 생성

In [37]:
# 쿼리 엔진
query_engine = index.as_query_engine()

In [39]:
# 쿼리 실행
query = 'what is the title of the paper? let me know the title of the paper'
response = query_engine.query(query)

In [40]:
# 응답 출력
print(response)

The title of the paper is "Attention Is All You Need."


## 2. 로컬(내 PC)에 인덱스 저장

- 장기적인 활용을 위해 로컬 디렉토리에 저장하고 나중에 불러오는 과정 (인덱스 영속성)이 필요하다.
- 새로운 임베딩을 생성할 필요가 없어서 기존 벡터 데이터를 그대로 활용할 수 있으므로 속도와 리소스 절약할 수 있다.

In [42]:
# Vector DB 관련 라마 인덱스 라이브러리 불러오기
from llama_index.core import StorageContext, load_index_from_storage

# 저장할 디렉토리 지정
persist_dir = './saved_index'

# 인덱스 저장
index.storage_context.persist(persist_dir=persist_dir)

In [43]:
# 저장된 인덱스를 불러오기 준비
storage_context = StorageContext.from_defaults(persist_dir=persist_dir)

# 저장된 인덱스를 로드(불러오기)
loaded_index = load_index_from_storage(storage_context=storage_context)

In [45]:
# 불러온 인덱스로 쿼리 엔진 생성
loaded_query_engine = loaded_index.as_query_engine()

# 다른 쿼리 실행
query = '이 논문에서 제안하는 모델의 장점은 무엇인가요?'
response = loaded_query_engine.query(query)

In [46]:
# 질문
print(f'질문: {query}')
print('-'*80)

# 답변
print(f'답변: {response}')

질문: 이 논문에서 제안하는 모델의 장점은 무엇인가요?
--------------------------------------------------------------------------------
답변: 이 논문에서 제안하는 모델의 장점은, 기존의 모델들보다 병렬 처리가 용이하고, 학습 및 추론 속도가 빠르다는 점입니다. 또한, 긴 문맥 정보를 효과적으로 처리할 수 있어 다양한 자연어 처리 작업에서 우수한 성능을 보입니다.


In [ ]:
## 돌발 과제!
# data 폴더 안 html만 노드파서
# data 폴더 안 마크다운(md)만 노드파서